# GPT-2 + Sky130 SAR qualification on a Colab T4

This notebook runs the pinned numerical workload with the guarded SAR dispatch policy. It does not claim analog hardware speed, energy, yield, or silicon qualification.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), 'CUDA is required; refusing to run a CPU substitute'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone https://github.com/mehtama1234/ai-hardware-analysis.git /content/ai-hardware-analysis
!pip install -q torch transformers huggingface_hub

In [ ]:
!python3 /content/ai-hardware-analysis/analog-in-memory-ai-inference/software-architecture/colab/run_profile_driven_gpt2_colab.py \
  --repo-root /content/ai-hardware-analysis \
  --output /content/gpt2-sar-t4 \
  --fetch-model --execute-model --device cuda

In [ ]:
!python3 /content/ai-hardware-analysis/analog-in-memory-ai-inference/software-architecture/scripts/check_profile_driven_colab_receipt.py \
  --require-cuda /content/gpt2-sar-t4/colab-receipt.json
!python3 /content/ai-hardware-analysis/analog-in-memory-ai-inference/software-architecture/scripts/check_gpt2_hybrid_evaluation.py \
  --package /content/gpt2-sar-t4/model-evaluation

In [ ]:
import json
from pathlib import Path
receipt = json.loads(Path('/content/gpt2-sar-t4/colab-receipt.json').read_text())
evaluation = json.loads(Path('/content/gpt2-sar-t4/model-evaluation/evaluation.json').read_text())
print('device:', evaluation['runtime']['device'])
for variant in evaluation['variants']:
    q = variant['quality']
    print(variant['id'], q['teacher_forced_argmax_agreement'], q['generation_exact_match_count'], q['nll_increase_nats'])
print('dispatch:', receipt['dispatch_policy']['status'])